<a href="https://colab.research.google.com/github/TheZaraKhan/ai-reliability/blob/main/notebooks/bike_share_reliability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How reliable are AI data-analysis assistants?

This notebook evaluates whether language models can answer questions about
Toronto Bike Share trips by writing pandas code.

## Stage 1 — Data source and quality checks

Source: City of Toronto Open Data, Bike Share Toronto Ridership Data.
Licence: Open Government Licence – Toronto.
Scope: Trips that started from January 1 through March 31, 2024.

In [9]:
import io
import json
import re
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd
from IPython.display import display

# Start with a tiny dataset whose expected results we can check by hand.
# Change this to False when you're ready to use the official Toronto data.
USE_FIXTURE = False

# Include January 1 and exclude April 1.
QUARTER_START = pd.Timestamp("2024-01-01")
QUARTER_END = pd.Timestamp("2024-04-01")

# CKAN is the API Toronto uses to list files in its open-data catalogue.
CKAN_URL = (
    "https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action/"
    "package_show?id=bike-share-toronto-ridership-data"
)
ARCHIVE_NAME = "bikeshare-ridership-2024.zip"

# An empty value after the final comma represents a missing station name.
# Trip 11 is duplicated. Trip 12 starts in March and ends in April.
FIXTURE = """Trip_Id,Trip_Duration,Start_Time,End_Time,End_Station_Name
10,60,2023-12-31 23:59:30,2024-01-01 00:00:30,Station A
11,59,2024-01-01 00:00:00,2024-01-01 00:00:59,
11,59,2024-01-01 00:00:00,2024-01-01 00:00:59,
12,100000,2024-03-31 23:59:59,2024-04-02 03:46:39,Station B
13,60,2024-04-01 00:00:00,2024-04-01 00:01:00,Station B
"""

In [10]:
from shutil import copyfileobj


def download_archive(destination: Path) -> None:
    """Find the official 2024 file through CKAN and save it locally."""

    # Ask the catalogue which files belong to the Bike Share dataset.
    with urlopen(CKAN_URL, timeout=60) as response:
        package = json.load(response)

    if not package["success"]:
        raise RuntimeError("The City of Toronto CKAN request failed.")

    # Find the download URL for the 2024 ZIP.
    resources = package["result"]["resources"]
    resource = next(
        (item for item in resources if item["name"] == ARCHIVE_NAME),
        None,
    )
    if resource is None:
        raise RuntimeError(f"CKAN did not list {ARCHIVE_NAME}.")

    # Use a temporary filename so an interrupted download can't be
    # mistaken for the finished archive.
    partial = destination.with_suffix(".part")
    try:
        with urlopen(resource["url"], timeout=120) as response:
            with partial.open("wb") as output:
                copyfileobj(response, output)

        # Make sure the downloaded ZIP contains a CSV before renaming it.
        with ZipFile(partial) as archive:
            if not any(name.lower().endswith(".csv") for name in archive.namelist()):
                raise ValueError("The archive contains no CSV.")

        partial.replace(destination)
    finally:
        # Remove the temporary file if the download or archive check failed.
        partial.unlink(missing_ok=True)

In [11]:
def snake_case(name: str) -> str:
    """Convert a header like 'Start Time' or 'Start_Time' to 'start_time'."""
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def load_quarter(source) -> pd.DataFrame:
    """Read a CSV and return trips that started in the selected quarter."""

    # Read just the header first and show exactly how each name changes.
    original = pd.read_csv(source, nrows=0).columns.tolist()
    normalized = [snake_case(name) for name in original]
    display(pd.DataFrame({
        "original_header": original,
        "normalized_header": normalized,
    }))

    # Stop with a useful message if the file has a different schema.
    required = {"trip_id", "trip_duration", "start_time", "end_time"}
    missing = required - set(normalized)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    if len(normalized) != len(set(normalized)):
        raise ValueError("Two source columns normalize to the same name.")

    # Rewind after reading the header, then process the large CSV in batches.
    source.seek(0)
    quarter_chunks = []

    for chunk in pd.read_csv(source, chunksize=200_000):
        chunk.columns = normalized

        # Convert start times from text to datetimes so we can compare dates.
        # An unexpected date format raises an error you can inspect.
        starts = pd.to_datetime(
            chunk["start_time"],
            format="%Y-%m-%d %H:%M:%S",
            errors="raise",
        )

        # The start time decides which quarter a trip belongs to.
        in_quarter = starts.ge(QUARTER_START) & starts.lt(QUARTER_END)
        if not in_quarter.any():
            continue

        # Keep every row in the quarter, including short and duplicate trips.
        selected = chunk.loc[in_quarter].copy()
        selected["start_time"] = starts.loc[in_quarter]
        selected["end_time"] = pd.to_datetime(
            selected["end_time"],
            format="%Y-%m-%d %H:%M:%S",
            errors="raise",
        )
        quarter_chunks.append(selected)

    if not quarter_chunks:
        raise ValueError("No trips found for the selected quarter.")

    # Join the selected batches into one DataFrame with a fresh row index.
    return pd.concat(quarter_chunks, ignore_index=True)

In [12]:


if USE_FIXTURE:
    print("Using synthetic practice data.")
    # StringIO lets pandas read the practice text like a CSV file.
    df = load_quarter(io.StringIO(FIXTURE))
else:
    print("Using City of Toronto data for 2024 Q1.")
    # Reuse the ZIP if it's already in the current Colab session.
    archive_path = Path(ARCHIVE_NAME)
    if not archive_path.exists():
        download_archive(archive_path)

    with ZipFile(archive_path) as archive:
        # Inspect the archive instead of guessing the CSV filename.
        csv_names = [
            name for name in archive.namelist() if name.lower().endswith(".csv")
        ]
        print("CSV files in archive:", csv_names)

        if len(csv_names) != 1:
            raise ValueError("Expected one CSV. Check the filenames printed above.")

        with archive.open(csv_names[0]) as source:
            df = load_quarter(source)

# head() previews the first five rows. info() shows data types and non-null counts.
display(df.head())
df.info()

Using City of Toronto data for 2024 Q1.
CSV files in archive: ['bikeshare-ridership-2024.csv']


,original_header,normalized_header
0,Trip_Id,trip_id
1,Trip_Duration,trip_duration
2,Start_Station_Id,start_station_id
3,Start_Time,start_time
4,Start_Station_Name,start_station_name
5,End_Station_Id,end_station_id
6,End_Time,end_time
7,End_Station_Name,end_station_name
8,Bike_Id,bike_id
9,User_Type,user_type


,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model
0,26682740,528,7041,2024-01-01 00:00:08,Edward St / Yonge St,7130.0,2024-01-01 00:08:56,Davenport Rd / Bedford Rd,7855,Casual,EFIT G5
1,26682742,1924,7245,2024-01-01 00:02:04,Tecumseth St / Queen St W - SMART,7259.0,2024-01-01 00:34:08,Lower Spadina Ave / Lake Shore Blvd W,2453,Casual,ICONIC
2,26682743,2741,7052,2024-01-01 00:03:25,Wellington St W / Bay St,7366.0,2024-01-01 00:49:06,Fort York Blvd / Bathurst St - SMART,7332,Member,ICONIC
3,26682744,353,7041,2024-01-01 00:03:39,Edward St / Yonge St,7418.0,2024-01-01 00:09:32,College Park - Yonge St Entrance,782,Member,ICONIC
4,26682746,2118,7721,2024-01-01 00:04:44,Widmer St / King St W,7721.0,2024-01-01 00:40:02,Widmer St / King St W,1008,Member,ICONIC


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 779266 entries, 0 to 779265
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   trip_id             779266 non-null  int64         
 1   trip_duration       779266 non-null  int64         
 2   start_station_id    779266 non-null  int64         
 3   start_time          779266 non-null  datetime64[ns]
 4   start_station_name  779266 non-null  object        
 5   end_station_id      778983 non-null  float64       
 6   end_time            779251 non-null  datetime64[ns]
 7   end_station_name    778983 non-null  object        
 8   bike_id             779266 non-null  int64         
 9   user_type           779266 non-null  object        
 10  bike_model          779266 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(4), object(4)
memory usage: 65.4+ MB


In [13]:
# Read the recorded duration as a number of seconds.
duration = pd.to_numeric(df["trip_duration"], errors="raise")

# These values are calculated from whichever DataFrame you loaded.
quality = pd.DataFrame(
    [
        ("Rows", len(df)),
        ("Minimum duration (seconds)", duration.min()),
        ("Maximum duration (seconds)", duration.max()),
        ("Trips under 60 seconds", int(duration.lt(60).sum())),
        ("Duplicate rows after the first", int(df.duplicated().sum())),
        (
            "Repeated non-missing trip IDs after the first",
            int(df["trip_id"].dropna().duplicated().sum()),
        ),
    ],
    columns=["check", "value"],
)

# isna() marks missing cells. Summing those marks gives a count per column.
missing_values = df.isna().sum().rename("null_count").to_frame()

display(quality)
display(missing_values)

# These expected values come from the five practice rows above.
# Run these checks only in fixture mode.
if USE_FIXTURE:
    assert df["trip_id"].tolist() == [11, 11, 12]
    assert pd.api.types.is_datetime64_any_dtype(df["start_time"])
    assert pd.api.types.is_datetime64_any_dtype(df["end_time"])
    assert df["end_time"].max() == pd.Timestamp("2024-04-02 03:46:39")
    assert duration.min() == 59
    assert duration.max() == 100000
    assert int(duration.lt(60).sum()) == 2
    assert int(df.duplicated().sum()) == 1
    assert int(df["trip_id"].dropna().duplicated().sum()) == 1
    assert missing_values.loc["end_station_name", "null_count"] == 2
    print("Stage 1 fixture checks passed.")

,check,value
0,Rows,779266
1,Minimum duration (seconds),0
2,Maximum duration (seconds),2320126
3,Trips under 60 seconds,1572
4,Duplicate rows after the first,0
5,Repeated non-missing trip IDs after the first,0


,null_count
trip_id,0
trip_duration,0
start_station_id,0
start_time,0
start_station_name,0
end_station_id,283
end_time,15
end_station_name,283
bike_id,0
user_type,0
